# Data preparation for the first part of the project:

In [2]:
from pathlib import Path

dossier = Path('../datasets')

for fichier in dossier.iterdir():
    if fichier.is_file(): 
        print(fichier.name)

Arkansas_Soybeans_boost4.csv
MCTNet_Final_58_California.csv
MCTNet_Final_136_California.csv
MCTNet_Final_171_California.csv
MCTNet_Final_72_California.csv
MCTNet_Arkansas_Soybeans_Seed123.csv
MCTNet_Final_162_California.csv
MCTNet_Arkansas_Others_Grass_Seed999.csv
MCTNet_Final_123_California.csv
MCTNet_BOOST_Soybeans_101_AR.csv
Arkansas_Soybeans_boost6.csv
MCTNet_Final_220_California.csv
MCTNet_Final_2_California.csv
MCTNet_California_Almonds_Seed42.csv
MCTNet_Final_86_California.csv
MCTNet_Final_188_California.csv
MCTNet_California_Almonds_500.csv
MCTNet_California_Others_Grass_Seed777.csv
MCTNet_BOOST_Soybeans_100_AR.csv
MCTNet_Final_141_California.csv
MCTNet_Final_169_California.csv
MCTNet_Final_10k_Arkansas.csv
MCTNet_Final_73_California.csv
MCTNet_Arkansas_Cotton_Seed2024.csv
Arkansas_Soybeans_boost2.csv
MCTNet_Final_60_California.csv
MCTNet_Final_227_California.csv
MCTNet_Final_207_California.csv
MCTNet_Final_163_California.csv
MCTNet_Arkansas_Others_Grass_400.csv
MCTNet_Final_14

### concat all csv files into one and drop duplicates

In [3]:
import pandas as pd
from pathlib import Path

dossier_source = Path('../datasets')
output_cali = 'california_full_unique.csv'
output_arka = 'arkansas_full_unique.csv'

def fusion_brute_dedoublonnee(motif, nom_sortie):
    # 1. Collecte de tous les fichiers
    fichiers = list(dossier_source.glob(f'*{motif}*.csv'))
    
    if not fichiers:
        print(f"⚠️ Aucun fichier trouvé pour {motif}")
        return

    print(f"📂 {motif} : Fusion de {len(fichiers)} fichiers en cours...")
    
    # 2. Lecture et concaténation
    liste_df = []
    for f in fichiers:
        try:
            df = pd.read_csv(f)
            # On ignore system:index car il n'est pas unique entre fichiers
            if 'system:index' in df.columns:
                df = df.drop(columns=['system:index'])
            liste_df.append(df)
        except Exception as e:
            print(f"❌ Erreur sur {f.name} : {e}")
    
    df_total = pd.concat(liste_df, ignore_index=True)
    nb_initial = len(df_total)

    # 3. DÉDOUBLONNAGE GÉOGRAPHIQUE
    # On utilise les bandes spectrales de la première date comme empreinte digitale.
    # Si d0_B2, B4, B8, B11 et B12 sont identiques, c'est le même pixel physique.
    signature_spectrale = ['d0_B2', 'd0_B4', 'd0_B8', 'd0_B11', 'd0_B12']
    
    # On s'assure que ces colonnes existent avant de filtrer
    cols_filtre = [c for c in signature_spectrale if c in df_total.columns]
    
    if cols_filtre:
        df_unique = df_total.drop_duplicates(subset=cols_filtre)
    else:
        # Si d0 n'existe pas, on prend les 5 premières colonnes numériques
        cols_filtre = df_total.select_dtypes(include=['number']).columns[:5]
        df_unique = df_total.drop_duplicates(subset=cols_filtre)

    nb_final = len(df_unique)

    # 4. Statistiques de sortie
    print(f"✅ Terminé pour {motif}:")
    print(f"   - Lignes brutes : {nb_initial}")
    print(f"   - Doublons supprimés : {nb_initial - nb_final}")
    print(f"   - Pixels uniques : {nb_final}")
    
    if 'cropland' in df_unique.columns:
        print("📊 Répartition par classe :")
        print(df_unique['cropland'].value_counts().sort_index())

    
    df_unique.to_csv(nom_sortie, index=False)
    print(f"💾 Sauvegardé sous : {nom_sortie}\n")

# Lancement
fusion_brute_dedoublonnee('California', output_cali)
fusion_brute_dedoublonnee('Arkansas', output_arka)

📂 California : Fusion de 153 fichiers en cours...
✅ Terminé pour California:
   - Lignes brutes : 35087
   - Doublons supprimés : 4675
   - Pixels uniques : 30412
📊 Répartition par classe :
cropland
1       179
2       349
3      3037
4         1
6        61
12        2
21       20
22       13
24      249
28        6
33      131
36     4282
37      101
41       28
42        1
43        6
44       12
49       28
51        3
54      461
58        5
59        1
61      378
66       33
68        7
69     6796
71        7
72      104
74        1
75     2743
76     3284
77       15
111       6
121     288
122     170
123      62
124      18
152      28
176    6515
195      56
204     655
205      33
206       2
208      23
209       2
211      18
213       3
215       2
217      53
219       2
220      10
225      64
226      12
227       1
228      43
236       2
Name: count, dtype: int64
💾 Sauvegardé sous : california_full_unique.csv

📂 Arkansas : Fusion de 70 fichiers en cours...
✅ Termin

### Re-order the columns:

In [32]:
import pandas as pd

def clean_and_reorder_columns(input_file, output_file):

    df = pd.read_csv(input_file)

    bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    
    ordered_cols = []
    for i in range(36):
        for b in bands:
            col_name = f'd{i}_{b}'
            if col_name in df.columns:
                ordered_cols.append(col_name)
    

    if 'cropland' in df.columns:
        ordered_cols.append('cropland')
    

    df_final = df[ordered_cols]
 
    df_final.to_csv(output_file, index=False)
    print(f"✅ Fichier ordonné sauvegardé : {output_file} (Colonnes : {len(ordered_cols)})")


try:
    clean_and_reorder_columns('california_full_unique.csv', 'california.csv')
    clean_and_reorder_columns('arkansas_full_unique.csv', 'arkansas.csv')
except FileNotFoundError as e:
    print(f"❌ Erreur : Assure-toi d'avoir téléchargé les fichiers du Drive. {e}")

✅ Fichier ordonné sauvegardé : california.csv (Colonnes : 361)
✅ Fichier ordonné sauvegardé : arkansas.csv (Colonnes : 361)


### California stats:

In [33]:
import pandas as pd


df = pd.read_csv('california.csv')


target_mapping = {
    69: 'Grapes',
    3:  'Rice',
    36: 'Alfalfa',
    75: 'Almonds',
    76: 'Pistachios'
}


objectifs = {
    'Grapes': 2054, 'Rice': 2037, 'Alfalfa': 974, 
    'Almonds': 783, 'Pistachios': 640, 'Others': 3512
}

counts = df['cropland'].value_counts().to_dict()


stats_cibles = {}
others_total = 0
others_details = {}

for code, count in counts.items():
    if code in target_mapping:
        stats_cibles[target_mapping[code]] = count
    else:
        
        others_total += count
        others_details[code] = count

stats_cibles['Others'] = others_total


print(f"{'Crop':<15} | {'Actuel':<8} | {'Objectif':<8} | {'Manquant':<8}")
print("-" * 50)

for name in ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']:
    actual = stats_cibles.get(name, 0)
    target = objectifs.get(name, 0)
    diff = target - actual
    status = f"{diff}" if diff > 0 else "OK"
    print(f"{name:<15} | {actual:<8} | {target:<8} | {status:<8}")

print("-" * 50)
print(f"TOTAL ACTUEL : {sum(stats_cibles.values())}")


Crop            | Actuel   | Objectif | Manquant
--------------------------------------------------
Grapes          | 6796     | 2054     | OK      
Rice            | 3037     | 2037     | OK      
Alfalfa         | 4282     | 974      | OK      
Almonds         | 2743     | 783      | OK      
Pistachios      | 3284     | 640      | OK      
Others          | 10270    | 3512     | OK      
--------------------------------------------------
TOTAL ACTUEL : 30412


### Arkansas stats:

In [34]:
import pandas as pd


df_ark = pd.read_csv('arkansas.csv')

target_mapping_ark = {
    1: 'Soybeans',
    3: 'Rice',
    5: 'Corn',
    2: 'Cotton'
}


objectifs_ark = {
    'Soybeans': 4677, 
    'Rice': 2423, 
    'Corn': 1522, 
    'Cotton': 762, 
    'Others': 616
}


counts_ark = df_ark['cropland'].value_counts().to_dict()


stats_ark = {}
others_total_ark = 0
others_details_ark = {}

for code, count in counts_ark.items():
    if code in target_mapping_ark:
        stats_ark[target_mapping_ark[code]] = count
    else:
        
        others_total_ark += count
        others_details_ark[code] = count

stats_ark['Others'] = others_total_ark


print(f"{'Crop':<15} | {'Actuel':<8} | {'Objectif':<8} | {'Manquant':<8}")
print("-" * 50)

for name in ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']:
    actual = stats_ark.get(name, 0)
    target = objectifs_ark.get(name, 0)
    diff = target - actual
    status = f"{diff}" if diff > 0 else "OK"
    print(f"{name:<15} | {actual:<8} | {target:<8} | {status:<8}")

print("-" * 50)
print(f"TOTAL ACTUEL : {sum(stats_ark.values())}")


Crop            | Actuel   | Objectif | Manquant
--------------------------------------------------
Soybeans        | 10341    | 4677     | OK      
Rice            | 10168    | 2423     | OK      
Corn            | 10640    | 1522     | OK      
Cotton          | 3199     | 762      | OK      
Others          | 2277     | 616      | OK      
--------------------------------------------------
TOTAL ACTUEL : 36625
